In [ ]:

# Shared setup — run this cell first
import re
import urllib.request
from pathlib import Path

import pandas as pd

from labeling.html_parser import parse as parse_html
from labeling.labeling_functions import (
    ABSTAIN, ALL_LFS, LABEL_NAMES,
    lf_author_a, lf_author_by_prefix, lf_author_classname,
    lf_author_itemprop, lf_author_rel, lf_author_schema, lf_author_schema_fuzzy, lf_author_xpath,
    lf_ingredient_kaggle, lf_ingredient_li_in_ul,
    lf_ingredient_measure, lf_ingredient_schema, lf_ingredient_xpath,
    lf_step_kaggle, lf_step_ol_li, lf_step_long_p,
    lf_step_schema, lf_step_xpath,
)
from labeling.schema_org import extract as extract_schema

CRAWL_LOG = Path('data/raw_html/crawl_log.csv')
KAGGLE_CSV = Path('data/kaggle/sampled_urls.csv')
N_SAMPLE = 10
_USER_AGENT = 'Mozilla/5.0 (compatible; recipe-scooper-test/1.0)'

_CONTROL_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def _sanitize(text):
    return _CONTROL_RE.sub('', text)

def _fetch_url(url):
    req = urllib.request.Request(url, headers={'User-Agent': _USER_AGENT})
    with urllib.request.urlopen(req, timeout=15) as resp:
        return resp.read().decode('utf-8', errors='replace')

def _build_node_df(html, kaggle_row):
    schema = extract_schema(html)
    nodes = parse_html(html)
    if not nodes:
        return pd.DataFrame()
    rows = []
    for node in nodes:
        rows.append({
            'text': _sanitize(node.text),
            'xpath': node.xpath,
            'tag': node.tag,
            'class_name': node.class_name or '',
            'link_href': node.link_href or '',
            'link_rel': node.link_rel or '',
            'itemprop': node.itemprop or '',
            'schema_name': schema.name or '',
            'schema_author': schema.author or '',
            'schema_date': schema.date_published or '',
            'schema_ingredients': '||'.join(schema.ingredients),
            'schema_instructions': '||'.join(schema.instructions),
            'kaggle_title': str(kaggle_row.get('title', '') or ''),
            'kaggle_ingredients': str(kaggle_row.get('ingredients', '') or ''),
            'kaggle_directions': str(kaggle_row.get('directions', '') or ''),
        })
    return pd.DataFrame(rows)

def _print_nodes(label, node_df, lfs):
    fired_any = False
    for _, row in node_df.iterrows():
        results = [(lf.name, LABEL_NAMES[lf(row)]) for lf in lfs if lf(row) != ABSTAIN]
        if not results:
            continue
        if not fired_any:
            print(f'--- {label} ---')
            fired_any = True
        print(f'  xpath : {row["xpath"]}')
        print(f'  tag   : <{row["tag"]}>')
        print(f'  text  : {row["text"][:200]}')
        for lf_name, label_name in results:
            print(f'    -> {lf_name}: {label_name}')
        print()
    if not fired_any:
        print(f'--- {label} ---')
        print('  [INFO] No nodes fired for this category\n')

def run_category(category_name, lfs, n=N_SAMPLE, index=None, url=None, file=None):
    if file is not None:
        file = Path(file)
        print(f'=== LF Category: {category_name.upper()} | {len(lfs)} LFs | local file ===\n')
        html = file.read_text(encoding='utf-8', errors='replace')
        node_df = _build_node_df(html, pd.Series())
        if node_df.empty:
            print('  [SKIP] No nodes extracted\n')
            return
        _print_nodes(str(file), node_df, lfs)
        return

    if url is not None:
        print(f'=== LF Category: {category_name.upper()} | {len(lfs)} LFs | fetching live URL ===\n')
        print(f'Fetching {url} ...')
        html = _fetch_url(url)
        node_df = _build_node_df(html, pd.Series())
        if node_df.empty:
            print('  [SKIP] No nodes extracted\n')
            return
        _print_nodes(url, node_df, lfs)
        return

    log_df = pd.read_csv(CRAWL_LOG)
    log_df = log_df[log_df['path'].notna() & (log_df['path'] != '')].reset_index(drop=True)

    if KAGGLE_CSV.exists():
        kaggle_df = pd.read_csv(KAGGLE_CSV).set_index('url')
    else:
        kaggle_df = pd.DataFrame()

    if index is not None:
        sample = log_df.iloc[[index]]
    else:
        sample = log_df.sample(min(n, len(log_df)))

    print(f'=== LF Category: {category_name.upper()} | {len(lfs)} LFs | {len(sample)} file(s) ===\n')

    for _, log_row in sample.iterrows():
        row_url = log_row['url']
        html_path = Path(log_row['path'])
        kaggle_row = kaggle_df.loc[row_url] if row_url in kaggle_df.index else pd.Series()
        try:
            html = html_path.read_text(encoding='utf-8', errors='replace')
        except OSError:
            continue
        node_df = _build_node_df(html, kaggle_row)
        if node_df.empty:
            continue
        _print_nodes(row_url, node_df, lfs)

print('Setup complete.')


In [ ]:
# AUTHOR LFs (8 total)
run_category('author', [
    lf_author_schema,
    lf_author_schema_fuzzy,
    lf_author_xpath,
    lf_author_classname,
    lf_author_a,
    lf_author_rel,
    lf_author_itemprop,
    lf_author_by_prefix,
])


In [3]:
# INGREDIENTS LFs: lf_ingredient_measure, lf_ingredient_xpath, lf_ingredient_schema, lf_ingredient_kaggle, lf_ingredient_li_in_ul
run_category('ingredients', [
    lf_ingredient_measure,
    lf_ingredient_xpath,
    lf_ingredient_schema,
    lf_ingredient_kaggle,
    lf_ingredient_li_in_ul,
])

=== LF Category: INGREDIENTS | 5 LFs | 10 file(s) ===

--- https://www.landolakes.com/recipe/725/pear-gorgonzola-focaccia ---
  xpath : /body[1]/div[2]/header[1]/nav[1]/div[1]/div[2]/ul[1]/li[3]/ul[1]/li[1]/div[1]/ul[1]/li[4]/a[1]/div[1]/span[1]
  tag   : <span>
  text  : Cheese
    -> lf_ingredient_schema: INGREDIENT
    -> lf_ingredient_kaggle: INGREDIENT

  xpath : /body[1]/div[2]/header[1]/nav[1]/div[1]/div[2]/ul[1]/li[3]/ul[1]/li[1]/div[1]/ul[1]/li[4]/a[1]/span[1]
  tag   : <span>
  text  : Cheese
    -> lf_ingredient_schema: INGREDIENT
    -> lf_ingredient_kaggle: INGREDIENT

  xpath : /body[1]/div[2]/main[1]/div[2]/div[1]/div[1]/ul[1]/li[3]/a[1]/span[1]
  tag   : <span>
  text  : PEAR
    -> lf_ingredient_schema: INGREDIENT
    -> lf_ingredient_kaggle: INGREDIENT

  xpath : /body[1]/div[2]/main[1]/div[3]/div[2]/div[1]/div[1]/p[1]
  tag   : <p>
  text  : frozen white bread dough, thawed
    -> lf_ingredient_schema: INGREDIENT
    -> lf_ingredient_kaggle: INGREDIENT

  xpath : /bo

In [9]:
# STEPS LFs: lf_step_xpath, lf_step_schema, lf_step_kaggle, lf_step_ol_li, lf_step_long_p
run_category('steps', [
    lf_step_xpath,
    lf_step_schema,
    lf_step_kaggle,
    lf_step_ol_li,
    lf_step_long_p,
])

=== LF Category: STEPS | 5 LFs | 10 file(s) ===

--- https://cooking.nytimes.com/recipes/9543 ---
  xpath : /body[1]/div[1]/div[1]/div[1]/div[2]/header[1]/div[1]/div[1]/nav[1]/ul[1]/li[3]/button[1]/span[1]
  tag   : <span>
  text  : Ingredients
    -> lf_step_schema: STEP
    -> lf_step_kaggle: STEP

  xpath : /body[1]/div[1]/div[1]/div[1]/div[2]/header[1]/div[1]/div[1]/nav[1]/ul[1]/li[6]/button[1]/span[1]
  tag   : <span>
  text  : About
    -> lf_step_schema: STEP
    -> lf_step_kaggle: STEP

  xpath : /body[1]/div[1]/div[1]/div[1]/div[2]/header[1]/div[1]/div[1]/nav[1]/div[1]/div[1]/div[1]/button[3]/span[1]
  tag   : <span>
  text  : Ingredients
    -> lf_step_schema: STEP
    -> lf_step_kaggle: STEP

  xpath : /body[1]/div[1]/div[1]/div[1]/div[2]/header[1]/div[1]/div[1]/nav[1]/div[1]/div[1]/div[1]/button[6]/span[1]
  tag   : <span>
  text  : About
    -> lf_step_schema: STEP
    -> lf_step_kaggle: STEP

  xpath : /body[1]/div[1]/div[1]/div[1]/div[2]/header[1]/div[1]/div[1]/nav[1]/di

In [11]:
# ALL LFs (all 19 across every category)
run_category('all', ALL_LFS)

=== LF Category: ALL | 19 LFs | 10 file(s) ===

--- https://cookpad.com/us/recipes/346834-cheese-ball-holiday-cheese-planet ---
  xpath : /body[1]/div[1]/nav[1]/div[2]/div[1]/div[2]/div[1]
  tag   : <div>
  text  : .
    -> lf_step_schema: STEP
    -> lf_step_kaggle: STEP

  xpath : /body[1]/div[1]/div[1]
  tag   : <div>
  text  : Main
    -> lf_step_schema: STEP
    -> lf_step_kaggle: STEP

  xpath : /body[1]/div[1]/div[1]/header[1]/div[1]/div[1]/div[1]/div[1]
  tag   : <div>
  text  : Cheese Ball Holiday Cheese Planet
    -> lf_title_schema: TITLE
    -> lf_title_kaggle: TITLE

  xpath : /body[1]/div[1]/div[1]/header[1]/div[1]/div[1]/div[1]/div[3]/div[1]/ul[1]/li[2]/div[1]/span[1]/form[1]/button[1]
  tag   : <button>
  text  : Unfollow
    -> lf_step_schema: STEP
    -> lf_step_kaggle: STEP

  xpath : /body[1]/div[1]/div[1]/header[1]/div[1]/div[1]/div[1]/div[3]/div[1]/ul[1]/li[6]/div[1]/a[1]
  tag   : <a>
  text  : Share
    -> lf_step_schema: STEP
    -> lf_step_kaggle: STEP

  xpat